# Aprendizado de Máquina — Lista prática 10

## KNN e Árvores de Classificação

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Esta é a lista que fecha o bloco de classificação, e o último exercício põe os
**dez** classificadores do curso lado a lado, nos mesmos dados, medidos por três
métricas. O que ele revela não é qual ganha:

> **um dos métodos fica em terceiro lugar em acurácia e é o pior de todos em
> calibração, por uma ordem de grandeza. Acertar a classe e acertar a
> probabilidade são competências separadas.**

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import math

import numpy as np
from matplotlib.pyplot import subplots

import sklearn.model_selection as skm
from sklearn.datasets import load_breast_cancer, make_moons
from sklearn.discriminant_analysis import (LinearDiscriminantAnalysis,
                                           QuadraticDiscriminantAnalysis)
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — as fronteiras do KNN

O conjunto `make_moons` são duas meias-luas entrelaçadas: a fronteira verdadeira
é curva e nenhum método linear dá conta. É o cenário em que o KNN brilha.

In [ ]:
X, y = make_moons(n_samples=400, noise=..., random_state=2026)   # (a)

X_tr, X_te, y_tr, y_te = skm.train_test_split(
    X, y, test_size=0.5, random_state=2026, stratify=y)

print("     k    treino    teste")
for k in (1, 5, 25, 50):
    modelo = KNeighborsClassifier(n_neighbors=...).fit(X_tr, y_tr)  # (b)
    print(f"  {k:4d}   {modelo.score(X_tr, y_tr):.4f}   {...:.4f}")   # (c)

> **Sua vez.** Desenhe as fronteiras de decisão para $k=1$ e $k=50$. Use
> `np.meshgrid` para gerar uma grade fina no plano, `modelo.predict` sobre ela, e
> `ax.pcolormesh(..., shading="auto", alpha=0.3)` para pintar o fundo. Por cima,
> os pontos de treino.

---
## Exercício 2 — Gini ou entropia?

A Lista Teórica 11 mostrou que o erro de classificação pode dar redução
**exatamente zero** num corte que separa um filho puro, e que por isso o
`scikit-learn` nem o oferece como critério de crescimento. Sobram Gini e
entropia. Quanto a escolha entre eles importa?

In [ ]:
dados = load_breast_cancer()
X_bc, y_bc = dados.data, dados.target
cv = skm.StratifiedKFold(5, shuffle=True, random_state=2026)

for criterio in (..., ...):                            # (a) e (b)
    for prof in (3, 5, None):
        arvore = DecisionTreeClassifier(criterion=criterio, max_depth=prof, random_state=0)
        acc = skm.cross_val_score(arvore, X_bc, y_bc, cv=cv).mean()
        print(f"  {criterio:8s} max_depth={str(prof):4s}: acuracia (CV) {acc:.4f}")

Confirme também que o erro de classificação **não** é uma opção.

In [ ]:
try:
    DecisionTreeClassifier(criterion=...).fit(X_bc, y_bc)   # (a)
except Exception as erro:
    print(type(erro).__name__)
    print(str(erro)[:150])

---
## Exercício 3 — o `max_features` que muda de padrão

Um detalhe do `scikit-learn` que costuma passar despercebido: o valor default de
`max_features` **não é o mesmo** na floresta de classificação e na de regressão.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

print("padrao no classificador:", ...)   # (a)
print("padrao no regressor:    ", RandomForestRegressor().max_features)

d = X_bc.shape[1]
print(f"\nd = {d}, sqrt(d) = {math.sqrt(d):.2f}  ->  a floresta de classificacao "
      f"sorteia {int(math.sqrt(d))} covariaveis por no")

In [ ]:
for mf in ("sqrt", 0.5, None):
    floresta = RandomForestClassifier(n_estimators=300, max_features=...,     # (a)
                                      random_state=0, n_jobs=-1)
    acc = skm.cross_val_score(floresta, X_bc, y_bc, cv=cv).mean()
    print(f"  max_features={str(mf):5s}: acuracia (CV) {acc:.4f}")

---
## Exercício 4 — os dez classificadores do curso

Todos os métodos de classificação vistos até aqui, no mesmo banco, com as mesmas
dobras, medidos por **três** métricas: acurácia (a decisão), AUC (a ordenação) e
Brier (a calibração). É a síntese das Aulas 07 a 11.

O `cross_val_predict` devolve, para cada observação, a probabilidade prevista
pelo modelo ajustado **sem ela** — é o que permite calcular o Brier honestamente.

In [ ]:
def tubo(modelo):
    return Pipeline([("escala", StandardScaler()), ("modelo", modelo)])


modelos = [
    ("logistica",  tubo(LogisticRegression(max_iter=5000))),
    ("LDA",        LinearDiscriminantAnalysis()),
    ("QDA",        QuadraticDiscriminantAnalysis()),
    ("GaussianNB", GaussianNB()),
    ("KNN (k=5)",  tubo(KNeighborsClassifier(5))),
    ("arvore",     DecisionTreeClassifier(max_depth=5, random_state=0)),
    ("floresta",   RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1)),
    ("AdaBoost",   AdaBoostClassifier(n_estimators=200, random_state=0)),
    ("SVM linear", tubo(SVC(kernel="linear", probability=True, random_state=0))),
    ("SVM RBF",    tubo(SVC(probability=True, random_state=0))),
]

print("  modelo          acuracia     AUC     Brier")
for nome, modelo in modelos:
    acc = skm.cross_val_score(modelo, X_bc, y_bc, cv=cv, scoring=...).mean()   # (a)
    auc = skm.cross_val_score(modelo, X_bc, y_bc, cv=cv, scoring=...).mean()    # (b)
    prob = skm.cross_val_predict(modelo, X_bc, y_bc, cv=cv,
                                 method=...)[:, 1]                        # (c)
    print(f"  {nome:14s}  {acc:.4f}   {auc:.4f}   {...:.4f}")  # (d)

> **Sua vez.** Passe o AdaBoost por uma calibração: envolva-o num
> `CalibratedClassifierCV(..., method="isotonic", cv=5)` e refaça a linha dele. A
> acurácia muda? E o Brier?

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | $k=1$ acerta 100% do treino e é o **pior** no teste (0,8650 contra 0,8950 de $k=5$) |
| 2 | Gini e entropia diferem no máximo 2,1 pontos; a profundidade decide muito mais |
| 2 | o erro de classificação nem consta entre os critérios aceitos pelo `scikit-learn` |
| 3 | `max_features="sqrt"` é o **pior** dos três valores neste banco (0,9578 contra 0,9614) |
| 4 | AdaBoost é 3º em acurácia e último em Brier, com $0{,}1480$ — **7×** a logística |
| 4 | a árvore isolada tem AUC 0,9294 por granularidade: 32 folhas, 32 probabilidades possíveis |

**A seguir.** Fecha o curso regular. As três aulas extras saem do aprendizado
supervisionado: E1 e E2 tratam de problemas **sem $Y$** — agrupar e reduzir
dimensão — e E3 aplica tudo a texto.